# AutoRA Workflow

Generated by AutoRA Workflow Editor on 2026-08-09T21:02:53.432Z

## 1. Install dependencies

In [ ]:
%pip install autora-core==5.0.3 autora-synthetic==2.2.0 autora-theorist-bms==1.0.6

## 2. Imports

In [ ]:
from autora.state import on_state, Delta, estimator_on_state, StandardState
from autora.variable import VariableCollection
from autora.experimentalist.grid import pool as grid_pooler
from autora.experimentalist.random import sample as random_sampler
from autora.experiment_runner.synthetic.economics.expected_value_theory import expected_value_theory
from autora.theorist.bms.regressor import BMSRegressor

import pandas as pd

## 3. Component definitions

In [ ]:
# Grid Pooler
@on_state()
def grid_pooler_on_state(variables: VariableCollection) -> Delta:
    return Delta(conditions=grid_pooler(variables))

In [ ]:
# Random Sampler
@on_state()
def random_sampler_on_state(conditions: pd.DataFrame, num_samples: int = 1) -> Delta:
    return Delta(conditions=random_sampler(conditions=conditions, num_samples=num_samples, replace=False))

In [ ]:
# Expected Value Theory (Synthetic, Economics)
@on_state()
def expected_value_theory_on_state(conditions: pd.DataFrame) -> Delta:
    runner = expected_value_theory(choice_temperature=0.1, value_lambda=0.5, resolution=10, minimum_value=-1, maximum_value=1)
    assert runner.run is not None
    return Delta(experiment_data=runner.run(conditions=conditions, added_noise=0.01))

In [ ]:
# BMS Regressor
bms_regressor_on_state = estimator_on_state(BMSRegressor(epochs=1500))

## 4. Run the workflow

In [ ]:
# Variables are created and governed by the experiment runner
runner = expected_value_theory(choice_temperature=0.1, value_lambda=0.5, resolution=10, minimum_value=-1, maximum_value=1)
assert runner.variables is not None
variables = runner.variables

# Initialize state
state = StandardState(variables=variables)

# Main experiment loop (1 cycles)
for i in range(1):
    print(f'Cycle {i}')

    # Grid Pooler
    state = grid_pooler_on_state(state)

    # Random Sampler
    state = random_sampler_on_state(state, num_samples=1)

    # Expected Value Theory (Synthetic, Economics)
    state = expected_value_theory_on_state(state)

    # BMS Regressor
    state = bms_regressor_on_state(state)


print("Workflow completed!")
state